# Deeper investigation — A daily report

**Worked solution** · [All exercises](../../index.html) · [Setup](../../README.md)

## What you’ll learn

- Group sales by both date and category.
- Calculate multiple aggregates for each group and explain the different questions they answer.

Optional. Complete [Exercise 4](../04-join-aggregate.ipynb) and its **Save and finish** cell first. This investigation uses the same saved work; it does not replace your core pipeline.

Completed answers use a separate solution workspace and do not replace participant work.

Run the supplied setup first. End with **Save and finish**; the next notebook loads your saved functions, so this kernel can be closed.

## Setup — supplied

Select the lab's `.venv` kernel. Stop Spark in the previous notebook before closing it. This uses the `create_spark` helper explained in [Exercise 0](../00-spark-session.ipynb). Missing earlier work? Use an explicit [catch-up step](../../RECOVERY.md).

In [1]:
from pathlib import Path
import sys

# Support opening the complete repository or its labs folder in VS Code.
LAB_ROOT = next(
    (candidate for parent in (Path.cwd(), *Path.cwd().parents)
     for candidate in (parent, parent / 'labs')
     if (candidate / 'workshop_runtime.py').is_file()),
    None,
)
if LAB_ROOT is None:
    raise FileNotFoundError('Open the complete labs project in VS Code; a notebook alone is not enough.')
if str(LAB_ROOT) not in sys.path:
    sys.path.insert(0, str(LAB_ROOT))

from uuid import uuid4

from pyspark.sql import Column, DataFrame
from pyspark.sql import functions as F

import lab_checks as check
from arrival_files import publish_arrival
from lab_checks import todo
from lab_workspace import Workspace
from workshop_runtime import DATA_ROOT, create_spark, finish_query, new_run, spark_path

workspace = Workspace(solutions=True)
product_key, clean_products, clean_sales, accepted_sales, rejected_sales, enrich_sales, category_totals = workspace.load('product_key', 'clean_products', 'clean_sales', 'accepted_sales', 'rejected_sales', 'enrich_sales', 'category_totals')
RUN_ROOT = new_run()
spark = create_spark(RUN_ROOT)
raw = spark.read.parquet(spark_path(DATA_ROOT / "sales.parquet"))
raw_products = spark.read.parquet(spark_path(DATA_ROOT / "products.parquet"))
products = clean_products(raw_products)
cleaned = clean_sales(raw)
accepted = accepted_sales(cleaned)
rejected = rejected_sales(cleaned)
enriched = enrich_sales(accepted, products)
report = category_totals(enriched)
print(f"Spark {spark.version}; inputs: {DATA_ROOT.name}; notebook ready")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/22 19:45:03 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.2.0; inputs: data; notebook ready


---
<a id="extension-daily"></a>
## Your task

Use `enriched` to produce `daily`: add `sale_date` from `sold_at`, group by date and category, and sum amounts as `total`. Also calculate `largest_sale` with a suitable aggregate.

You should have four date/category rows whose totals still sum to 100.00. Explain how `largest_sale` differs from `total`.

In [2]:
daily = (
    enriched.withColumn("sale_date", F.to_date("sold_at"))
    .groupBy("sale_date", "category")
    .agg(F.sum("amount").alias("total"), F.max("amount").alias("largest_sale"))
)
daily.orderBy("sale_date", "category").show()

+----------+--------+-----+------------+
| sale_date|category|total|largest_sale|
+----------+--------+-----+------------+
|2026-09-01|   books|25.00|       25.00|
|2026-09-01|   games|40.00|       40.00|
|2026-09-02|   books|25.00|       15.00|
|2026-09-02|unmapped|10.00|       10.00|
+----------+--------+-----+------------+



In [3]:
check.daily(daily)

Daily report verified: four groups, total 100.00.


<details><summary>Hint</summary>

Convert the timestamp with `to_date` before grouping. Each aggregate answers a different question about the sales in that group.

</details>

<a id="finish"></a>
## Save and finish

Run once the core checks pass, whether or not you did the optional section. This saves your functions or stream handoff, then stops this notebook’s queries and Spark. Your work remains in `learner_work/`.

In [4]:
for active_query in spark.streams.active:
    active_query.stop()
spark.stop()
print("Session stopped; exercise files are under", RUN_ROOT.relative_to(LAB_ROOT))

Session stopped; exercise files are under runs/run-a42ffe2177


Return to [all exercises](../../index.html).